In [2]:
import os # for file and directory operations
import pandas as pd # for data manipulation
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM # for tokenization and model loading
from sklearn.model_selection import train_test_split # for splitting the dataset
import re # support for regex

directory = r"../data/raw/finance/" # directory containing the CSV files

cleaned_dfs = [] # list to hold data from all CSV files

# check if directory exists
if not os.path.exists(directory):
    print(f"Directory '{directory}' does not exist.")
    exit()
else:
    # if the directory exists, process each CSV file
    print(f"Processing files in directory: {directory}")
    for filename in os.listdir(directory):
        if filename.endswith(".csv"):
            file_path = os.path.join(directory, filename)
            try:
                # read the CSV file
                df = pd.read_csv(file_path)
                # remove all columns except for Content and Summary
                df = df[['Content', 'Summary']]
                # drop empty rows
                df.dropna(how='any', inplace=True)
                # append the DataFrame to the list
                cleaned_dfs.append(df)
            except Exception as e:
                print(f"Error while processing file {filename}: {e}")

# function to remove encoding artifacts
def cleanText(text: str):
    if not isinstance(text, str):
        return " "
    
    # fix encoding issues
    text = text.encode("utf-8", "ignore").decode("utf-8", "ignore")

    # remove weird unicode characters
    text = text.replace("â€™", "'").replace("â€œ", '"').replace("â€", '"')
    text = text.replace("â€“", "-").replace("â€”", "-")
    text = text.replace("Ã", " ")

    # remove multiple spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

print("All CSV files processed.")

# concatenate all DFs into a single dataframe
if cleaned_dfs:
    combined_df = pd.concat(cleaned_dfs, ignore_index=True)
else: # odd scenario where no CSV files were found
    print("No valid CSV files found.")
    exit()

# apply the cleanText()
combined_df['Content'] = combined_df['Content'].apply(cleanText)
combined_df['Summary'] = combined_df['Summary'].apply(cleanText)

# Clean the Text in the merged DataFrame
# 1. Remove leading and trailing whitespace
combined_df['Content'] = combined_df['Content'].str.strip()
combined_df['Summary'] = combined_df['Summary'].str.strip()
# 2. Remove any unnecessary characters
combined_df['Content'] = combined_df['Content'].str.replace(r'\n+', '', regex=True) # new lines
combined_df['Summary'] = combined_df['Summary'].str.replace(r'\n+', '', regex=True) # new lines
combined_df['Content'] = combined_df['Content'].str.replace(r'<.*?>', '', regex=True) # HTML tags
combined_df['Summary'] = combined_df['Summary'].str.replace(r'<.*?>', '', regex=True) # HTML tags
combined_df['Content'] = combined_df['Content'].str.replace(r'\xa0', '', regex=True) # non-breaking spaces
combined_df['Summary'] = combined_df['Summary'].str.replace(r'\xa0', '', regex=True) # non-breaking spaces
combined_df['Content'] = combined_df['Content'].str.replace(r'http\S+', '', regex=True) # links
combined_df['Summary'] = combined_df['Summary'].str.replace(r'http\S+', '', regex=True) # links
# lowercase the text
combined_df['Content'] = combined_df['Content'].str.lower()
combined_df['Summary'] = combined_df['Summary'].str.lower()
# remove duplicate rows
combined_df.drop_duplicates(inplace=True)

# remove rows where Content or Summary is empty
combined_df = combined_df[(combined_df['Content'].str.strip() != '') & (combined_df['Summary'].str.strip() != '')]

# labelling rows with type of finance news article: (stock, general)
def labelNews(content: str):
    stock_keywords = [
        "stock", "share", "nse", "bse", "nasdaq", "dow jones",
        "sensex", "ipo", "dividend", "q1", "q2", "q3", "q4",
        "quarterly", "earnings", "profit", "loss", "%"
    ]

    if (keyword in content for keyword in stock_keywords):
        return "stock"
    return "general"

combined_df["Type"] = combined_df['Content'].apply(labelNews)

print(f"Combined DataFrame Head:\n{combined_df.head()}")

# Chunking the DataFrame into smaller parts
chunk_size_content = 1024 # size of each chunk
summary_chunk_size = 128

# tokenize the input text and check token length
tokenizer = AutoTokenizer.from_pretrained("../models/bart-base")

chunked_data = []

# iterate through the rows of the DataFrame
for _, row in combined_df.iterrows():
    content = row['Content']
    summary = row['Summary']
    article_type = row['Type']

    content_tokens = tokenizer.encode(content, truncation=False)
    summary_tokens = tokenizer.encode(summary, truncation=True, max_length=summary_chunk_size)
    token_length = len(content_tokens)

    print("Token Length: ", token_length)

    # if the content is too long, split it into chunks
    if token_length > chunk_size_content:
        # split the content into chunks
        content_chunks = [content_tokens[i:i+chunk_size_content] for i in range(0, len(content_tokens), chunk_size_content)]

        # create a new row for each chunk
        for content_chunk in content_chunks:
            chunked_data.append({
                'Content': tokenizer.decode(content_chunk, skip_special_tokens=True),
                'Summary': tokenizer.decode(summary_tokens, skip_special_tokens=True),
                'Type': article_type
            })
    else:
        chunked_data.append({
            'Content': tokenizer.decode(content_tokens, skip_special_tokens=True),
            'Summary': tokenizer.decode(summary_tokens, skip_special_tokens=True),
            'Type': article_type
        })

# create a new DataFrame from the chunked data
chunked_df = pd.DataFrame(chunked_data)

# 70-30 split of the chunked DataFrame
train_df, test_df = train_test_split(chunked_df, test_size=0.3, random_state=42)

os.makedirs("../data/cleaned", exist_ok=True)
train_df.to_csv("../data/cleaned/train.csv", index=False)
test_df.to_csv("../data/cleaned/test.csv", index=False)

print("Done. Train and test CSVs saved.")

d:\Programming\Project\News-Summarizer-LLM\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Processing files in directory: ../data/raw/finance/
All CSV files processed.
Combined DataFrame Head:
                                             Content  \
0  reuterspayments companies and banks are at log...   
1  agenciesthe bank presently has 20 million acti...   
2  ptikarnataka chief minister basavaraj bommai.k...   
3  getty imagesthe budget for fy22 had imposed in...   
4  agenciesthere is a view in the insurance indus...   

                                             Summary   Type  
0  reuterspayments companies and banks are at log...  stock  
1  agenciesthe bank presently has 20 million acti...  stock  
2  ptikarnataka chief minister basavaraj bommai.k...  stock  
3  getty imagesthe budget for fy22 had imposed in...  stock  
4  agenciesthere is a view in the insurance indus...  stock  
Token Length:  585
Token Length:  524
Token Length:  1180
Token Length:  496
Token Length:  270
Token Length:  703
Token Length:  341
Token Length:  588
Token Length:  944
Token Length:  95